---
title: Query via h5netcdf
---

# Query NISAR GUNW via h5netcdf (baseline)

For comparison, open the same NISAR granule the traditional way using
earthaccess and h5netcdf, and query the same 10 random points.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")

## Open with earthaccess + h5netcdf

In [ ]:
import earthaccess
import xarray as xr

earthaccess.login()

results = earthaccess.search_data(
    short_name="NISAR_L2_GUNW_BETA_V1",
    point=(174.1192, -39.3379),  # lon, lat
    temporal=("2026-01-01", "2026-01-04"),
)

files = earthaccess.open(results)
ds = xr.open_dataset(
    files[0],
    group="/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH",
    engine="h5netcdf",
    decode_timedelta=False,
    phony_dims="access",
)
ds

## Query the same 10 random points

In [ ]:
import numpy as np
import time

varname = "unwrappedPhase"

rng = np.random.default_rng(42)
ny, nx = ds.dims["yCoordinates"], ds.dims["xCoordinates"]
yi = rng.integers(0, ny, size=10)
xi = rng.integers(0, nx, size=10)

t0 = time.perf_counter()
values = ds[varname].values[yi, xi]
elapsed = time.perf_counter() - t0

print(f"Queried 10 points in {elapsed:.2f}s")

## Results

In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    {
        "y_index": yi,
        "x_index": xi,
        varname: values,
    }
)
results_df

## Visualize unwrapped phase

In [ ]:
import pygmt

fig = pygmt.Figure()
pygmt.makecpt(
    cmap="SCM/romaO", series=[0, 2 * np.pi, np.pi / 2], cyclic=True, continuous=True
)
fig.grdimage(grid=ds.unwrappedPhase, cmap=True, frame=True)
fig.colorbar()
fig.show()

## Visualize coherence magnitude

In [ ]:
ds.coherenceMagnitude.plot()

Note: `earthaccess.open` streams the file through HTTPS via fsspec.
Accessing `.values` may transfer a large portion of the file depending
on the HDF5 chunking. Compare the wall-clock time with the Icechunk
notebook, which fetches only the specific chunks containing the 10
query points.